# Subgradients and Proximal Methods

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 04.03 |
| Optional | 13.03 |
| Time | ~80 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/04_Optimization/10_subgradients_and_proximal_methods.ipynb)

---

## 🎯 Learning Objective

Extend gradient-based optimization beyond smooth functions: define subgradients for non-smooth
convex objectives, derive the proximal operator framework, and implement the ISTA and FISTA
algorithms that solve every L1-regularized problem in practice.

---

## 📐 Theory

`04.03` established the convergence theory for smooth convex optimization: when $f$ has a
Lipschitz-continuous gradient, gradient descent converges at $O(1/T)$ for convex $f$ and
linearly for $\mu$-strongly convex $f$. But many of the most important objectives in machine
learning are **not smooth** — the gradient simply does not exist at certain points — and the
entire machinery of `04.02`–`04.03` technically does not apply. This notebook develops the
mathematical tools that fill that gap.

### 1. Non-smooth functions in ML

Four workhorses of modern ML are non-differentiable at one or more points:

- **The $\ell_1$ norm** $\|\mathbf{w}\|_1 = \sum_j |w_j|$ — the Lasso/elastic-net regularizer.
  Each $|w_j|$ has no derivative at $w_j = 0$, exactly where sparsity lives.
- **The hinge loss** $\max(0,\, 1 - y\cdot f(\mathbf{x}))$ — the SVM objective (`04.05`'s dual
  derivation sidesteps the non-smoothness; here we confront it directly).
- **The ReLU activation** $\max(0, x)$ — non-differentiable at $x = 0$. Every ReLU network's
  loss is piecewise-linear in each pre-activation, hence globally non-smooth.
- **Total variation** $\sum_i |w_{i+1} - w_i|$ — a non-smooth penalty for signal denoising and
  image reconstruction.

Since the gradient $\nabla f(\mathbf{w})$ does not exist at the non-smooth points, gradient
descent as defined in `04.02` technically cannot take a step there. Three strategies exist:
(a) *smooth approximation* — replace $|w_j|$ with $\sqrt{w_j^2 + \epsilon}$ and hope
$\epsilon$ is small enough; (b) *subgradients* — generalize the gradient to non-smooth points;
(c) *proximal operators* — handle the non-smooth part in a separate, closed-form step. This
notebook develops (b) and (c) rigorously.

### 2. Subgradients and subdifferentials

**Definition.** A vector $\mathbf{g}$ is a **subgradient** of a convex function $f$ at
$\mathbf{x}$ if it defines a supporting hyperplane that lies everywhere below the function:

$$f(\mathbf{y}) \;\ge\; f(\mathbf{x}) + \mathbf{g}^\top(\mathbf{y} - \mathbf{x})
\qquad \forall\, \mathbf{y}$$

Compare this to the first-order condition for smooth convex functions (`04.03`): there the
gradient $\nabla f(\mathbf{x})$ is the *unique* vector satisfying this inequality. At a
non-smooth point, many vectors can satisfy it simultaneously.

The **subdifferential** $\partial f(\mathbf{x})$ is the set of *all* subgradients at
$\mathbf{x}$. It is always a closed, convex, non-empty set (for convex $f$ finite at
$\mathbf{x}$).

**Key properties:**

1. *Smooth case:* If $f$ is differentiable at $\mathbf{x}$, then
   $\partial f(\mathbf{x}) = \{\nabla f(\mathbf{x})\}$ — the subdifferential collapses to the
   singleton gradient. Subgradients genuinely generalize gradients.

2. *Optimality condition:* $\mathbf{x}^\star$ is a global minimizer of convex $f$ if and only
   if $\mathbf{0} \in \partial f(\mathbf{x}^\star)$. This is the non-smooth analogue of
   $\nabla f(\mathbf{x}^\star) = \mathbf{0}$.

3. *Canonical example:* For $f(x) = |x|$, the subdifferential is:
   $$\partial f(x) = \begin{cases} \{+1\} & x > 0 \\ [-1,\, +1] & x = 0 \\
   \{-1\} & x < 0 \end{cases}$$
   At $x = 0$ the subdifferential is the entire interval $[-1, 1]$: every slope in that range
   defines a valid supporting line for the V-shaped graph of $|x|$.

4. *Subdifferential calculus:*
   - **Sum rule:** $\partial(f + g)(\mathbf{x}) \supseteq \partial f(\mathbf{x}) +
     \partial g(\mathbf{x})$, with equality when both $f$ and $g$ are convex.
   - **Scaling:** $\partial(\alpha f)(\mathbf{x}) = \alpha\,\partial f(\mathbf{x})$ for
     $\alpha > 0$.
   - **Affine composition:** $\partial(f(A\mathbf{x} + \mathbf{b})) = A^\top
     \partial f(A\mathbf{x} + \mathbf{b})$ for a matrix $A$.

**Subgradient descent.** Replace the gradient with any subgradient:
$\mathbf{w}_{t+1} = \mathbf{w}_t - \eta_t \mathbf{g}_t$ where
$\mathbf{g}_t \in \partial f(\mathbf{w}_t)$.

*Convergence rates:*
- Convex, $G$-Lipschitz $f$, with step sizes $\eta_t = c/\sqrt{t}$:
  $$\min_{s \le T} f(\mathbf{w}_s) - f^\star \;\le\; O\!\left(\frac{G\,\|\mathbf{w}_0
  - \mathbf{w}^\star\|}{\sqrt{T}}\right)$$
- $\mu$-strongly convex: $O(G^2 / (\mu T))$ with $\eta_t = 1/(\mu t)$.

**Critical difference from gradient descent:** the objective is **not** guaranteed to decrease
at every step. A subgradient step can *increase* $f(\mathbf{w}_t)$ while still moving closer
to $\mathbf{w}^\star$ in Euclidean distance. Progress is measured in
$\|\mathbf{w}_t - \mathbf{w}^\star\|$, not $f(\mathbf{w}_t)$ — a fundamental conceptual shift
from the smooth theory in `04.03`. This is also why diminishing step sizes are *required*:
a fixed $\eta$ causes the iterates to oscillate around $\mathbf{w}^\star$ without converging.

### 3. Proximal operators

For a convex function $g$ (possibly non-smooth, possibly taking the value $+\infty$), the
**proximal operator** is:

$$\operatorname{prox}_{\lambda g}(\mathbf{v})
= \arg\min_{\mathbf{x}}\left\{g(\mathbf{x})
+ \frac{1}{2\lambda}\|\mathbf{x} - \mathbf{v}\|^2\right\}$$

Intuition: find the point that balances minimizing $g$ against staying close to $\mathbf{v}$.
The parameter $\lambda > 0$ controls the trade-off — larger $\lambda$ allows more movement
toward $\arg\min g$, smaller $\lambda$ keeps the result closer to $\mathbf{v}$.

**Four key examples:**

| $g$ | $\operatorname{prox}_{\lambda g}(\mathbf{v})$ | Name / connection |
| :--- | :--- | :--- |
| $g = 0$ | $\mathbf{v}$ | **Identity** — unconstrained GD is a special case |
| $g = \iota_C$ (indicator of convex set $C$) | $\operatorname{Proj}_C(\mathbf{v})$ | **Projection** — projected GD from `04.05` |
| $g = \|\cdot\|_1$ | $\mathcal{S}_\lambda(\mathbf{v})_j = \operatorname{sign}(v_j)\max(|v_j| - \lambda,\, 0)$ | **Soft-thresholding** — sparsity engine |
| $g = \frac{1}{2}\|\cdot\|_2^2$ | $\frac{\mathbf{v}}{1 + \lambda}$ | **Shrinkage** — Ridge/weight decay |

The soft-thresholding operator $\mathcal{S}_\lambda$ is the mathematical reason L1
regularization produces **exact zeros**: every coordinate with $|v_j| < \lambda$ is mapped to
exactly $0$, not merely shrunk toward it. Compare the shrinkage operator for $\ell_2$: it
scales all coordinates toward zero but never reaches it. This is the geometric distinction
between L1 and L2 regularization, made precise.

**Derivation of soft-thresholding.** For $g(x) = |x|$ (scalar case), the proximal problem is
$\min_x \{|x| + \frac{1}{2\lambda}(x - v)^2\}$. The subdifferential of the objective is
$\partial|x| + \frac{1}{\lambda}(x - v)$. Setting this to contain $0$:
- If $x > 0$: $1 + \frac{1}{\lambda}(x-v) = 0 \Rightarrow x = v - \lambda$. Valid when
  $v > \lambda$.
- If $x < 0$: $-1 + \frac{1}{\lambda}(x-v) = 0 \Rightarrow x = v + \lambda$. Valid when
  $v < -\lambda$.
- If $x = 0$: requires $v/\lambda \in [-1, 1]$, i.e. $|v| \le \lambda$.

Combining: $\operatorname{prox}_{\lambda|\cdot|}(v) = \operatorname{sign}(v)\max(|v|-\lambda,\,0)$.
For $\mathbb{R}^d$, separability of $\|\cdot\|_1 = \sum_j |\cdot_j|$ means the proximal
operator applies coordinate-wise.

### 4. Proximal gradient descent (ISTA)

Many ML objectives have **composite structure**: $\min_{\mathbf{w}} F(\mathbf{w}) =
f(\mathbf{w}) + g(\mathbf{w})$ where $f$ is smooth (differentiable, $L$-Lipschitz gradient)
and $g$ is convex but possibly non-smooth. The Lasso is the prototype:
$f(\mathbf{w}) = \frac{1}{2}\|X\mathbf{w} - \mathbf{y}\|^2$ (smooth least squares) and
$g(\mathbf{w}) = \lambda\|\mathbf{w}\|_1$ (non-smooth L1 penalty).

**The ISTA update** (Iterative Shrinkage-Thresholding Algorithm):

$$\mathbf{w}_{t+1} = \operatorname{prox}_{\eta g}\!\big(\mathbf{w}_t
- \eta\,\nabla f(\mathbf{w}_t)\big)$$

This alternates a gradient step on the smooth part $f$ with a proximal step on the non-smooth
part $g$. For the Lasso ($g = \lambda\|\cdot\|_1$), the proximal step is soft-thresholding:
take a gradient step on the least-squares loss, then threshold every coordinate. The algorithm
is remarkably simple — it reduces to two lines of code.

**Convergence.** For convex $f$ with $L$-Lipschitz gradient, step size $\eta = 1/L$:

$$F(\mathbf{w}_T) - F^\star \;\le\; \frac{L\,\|\mathbf{w}_0 - \mathbf{w}^\star\|^2}{2T}$$

This is $O(1/T)$ — the same rate as gradient descent on smooth problems (`04.03`). The
proximal step is **non-expansive**: $\|\operatorname{prox}(\mathbf{x}) -
\operatorname{prox}(\mathbf{y})\| \le \|\mathbf{x} - \mathbf{y}\|$, so it never hurts
convergence.

### 5. FISTA (Fast ISTA) — Nesterov acceleration for proximal gradient

Beck & Teboulle (2009) applied Nesterov's momentum trick to proximal gradient descent:

$$\mathbf{y}_t = \mathbf{w}_t + \frac{t_k - 1}{t_{k+1}}(\mathbf{w}_t - \mathbf{w}_{t-1}),
\qquad \mathbf{w}_{t+1} = \operatorname{prox}_{\eta g}\!\big(\mathbf{y}_t
- \eta\,\nabla f(\mathbf{y}_t)\big)$$

where $t_1 = 1$ and $t_{k+1} = \frac{1 + \sqrt{1 + 4t_k^2}}{2}$ (which gives the
momentum coefficient $\frac{t_k - 1}{t_{k+1}} \approx \frac{k-1}{k+2}$ for large $k$ —
compare `04.04`'s Nesterov momentum).

**Convergence:**

$$F(\mathbf{w}_T) - F^\star \;\le\; \frac{2L\,\|\mathbf{w}_0 - \mathbf{w}^\star\|^2}
{(T+1)^2}$$

This is $O(1/T^2)$ — a free quadratic speedup over ISTA from a simple extrapolation step.
FISTA is the standard fast solver for L1, elastic net, group Lasso, and all structured-sparse
problems. It is optimal: Nesterov (2004) proved that no first-order method can beat
$O(1/T^2)$ for convex smooth (or composite) problems — see `04.11` for the full rate
taxonomy.

### 6. Moreau envelope and Moreau decomposition

The **Moreau envelope** of $g$ with parameter $\lambda > 0$ is:

$$M_\lambda g(\mathbf{v}) = \min_{\mathbf{x}}\left\{g(\mathbf{x})
+ \frac{1}{2\lambda}\|\mathbf{x} - \mathbf{v}\|^2\right\}
= g\big(\operatorname{prox}_{\lambda g}(\mathbf{v})\big)
+ \frac{1}{2\lambda}\|\operatorname{prox}_{\lambda g}(\mathbf{v}) - \mathbf{v}\|^2$$

It is always smooth (even when $g$ is not) with gradient
$\nabla M_\lambda g(\mathbf{v}) = \frac{1}{\lambda}(\mathbf{v}
- \operatorname{prox}_{\lambda g}(\mathbf{v}))$, which is $1/\lambda$-Lipschitz. The Moreau
envelope "smooths" a non-smooth function while preserving its minimizers:
$\arg\min M_\lambda g = \arg\min g$.

The **Moreau decomposition** connects proximal operators to convex conjugates:

$$\mathbf{v} = \operatorname{prox}_{\lambda g}(\mathbf{v})
+ \lambda\,\operatorname{prox}_{g^\star/\lambda}(\mathbf{v}/\lambda)$$

where $g^\star$ is the convex conjugate (Fenchel dual) of $g$. This is the proximal analogue
of the orthogonal decomposition in linear algebra: every point decomposes into its proximal
image and a scaled proximal image under the conjugate. For $g = \|\cdot\|_1$, the conjugate
is $g^\star = \iota_{\|\cdot\|_\infty \le 1}$ (indicator of the $\ell_\infty$ unit ball),
so the decomposition becomes:
$\mathbf{v} = \mathcal{S}_\lambda(\mathbf{v}) + \lambda\,\operatorname{Proj}_{\|\cdot\|_\infty
\le 1}(\mathbf{v}/\lambda)$ — soft-thresholding plus projection, summing to the identity.
This duality is central to ADMM-based algorithms (forward reference to `04.12`).

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Three panels make the core ideas visible. *Left:* the subdifferential of $f(x) = |x|$ at
$x = 0$ — every line with slope in $[-1, 1]$ supports the graph from below, sweeping out the
entire interval of valid subgradients. *Center:* the soft-thresholding operator
$\mathcal{S}_\lambda(v)$ for several values of $\lambda$ — the dead zone $[-\lambda, \lambda]$
mapped to exactly zero is the sparsity mechanism. *Right:* ISTA vs FISTA convergence on a
Lasso problem — the $O(1/T)$ vs $O(1/T^2)$ rates are clearly visible on a log scale.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# --- Left: subgradient as supporting hyperplane at |x| = 0 ---
ax = axes[0]
x = np.linspace(-2, 2, 300)
ax.plot(x, np.abs(x), color='#4C72B0', lw=2.5, label=r'$f(x)=|x|$')
slopes = np.linspace(-1, 1, 9)
for i, g in enumerate(slopes):
    alpha = 0.3 if abs(g) < 0.99 else 0.8
    ax.plot(x, g * x, '--', color='#C44E52', lw=0.8, alpha=alpha,
            label=r'subgradients $g \in [-1,1]$' if i == 0 else None)
ax.set_xlim(-2, 2); ax.set_ylim(-1.5, 2)
ax.set_title(r'Subdifferential of $|x|$ at $x=0$: $\partial f(0) = [-1,1]$')
ax.legend(fontsize=8, loc='upper right')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')

# --- Center: soft-thresholding for several lambda ---
ax = axes[1]
v = np.linspace(-3, 3, 500)
for lam in [0.3, 0.8, 1.5]:
    st = np.sign(v) * np.maximum(np.abs(v) - lam, 0)
    ax.plot(v, st, lw=2, label=rf'$\lambda={lam}$')
ax.plot(v, v, ':', color='gray', lw=1, label='identity')
ax.axhline(0, color='black', lw=0.5); ax.axvline(0, color='black', lw=0.5)
ax.set_title(r'Soft-thresholding $\mathcal{S}_\lambda(v)$: dead zone creates exact zeros')
ax.set_xlabel('v'); ax.set_ylabel(r'$\mathcal{S}_\lambda(v)$')
ax.legend(fontsize=8)

# --- Right: ISTA vs FISTA convergence (quick demo) ---
ax = axes[2]
rng = np.random.default_rng(42)
n, d = 100, 50
X = rng.normal(size=(n, d))
w_true = np.zeros(d); w_true[:5] = rng.standard_normal(5) * 3
y = X @ w_true + rng.normal(scale=0.1, size=n)
lam = 0.5
L = np.linalg.eigvalsh(X.T @ X / n).max()
eta = 1.0 / L

def lasso_obj(w, X, y, lam):
    return 0.5 * np.mean((X @ w - y)**2) + lam * np.linalg.norm(w, 1)

def soft_thresh(v, t):
    return np.sign(v) * np.maximum(np.abs(v) - t, 0)

# ISTA
w_ista = np.zeros(d)
ista_vals = []
for _ in range(300):
    ista_vals.append(lasso_obj(w_ista, X, y, lam))
    grad = X.T @ (X @ w_ista - y) / n
    w_ista = soft_thresh(w_ista - eta * grad, eta * lam)

# FISTA
w_fista = np.zeros(d); w_prev = w_fista.copy(); t_k = 1.0
fista_vals = []
for k in range(300):
    fista_vals.append(lasso_obj(w_fista, X, y, lam))
    t_next = (1 + np.sqrt(1 + 4 * t_k**2)) / 2
    y_k = w_fista + (t_k - 1) / t_next * (w_fista - w_prev)
    grad = X.T @ (X @ y_k - y) / n
    w_prev = w_fista.copy()
    w_fista = soft_thresh(y_k - eta * grad, eta * lam)
    t_k = t_next

f_star = min(min(ista_vals), min(fista_vals))
ax.semilogy(np.array(ista_vals) - f_star + 1e-15, lw=2, label='ISTA $O(1/T)$', color='#C44E52')
ax.semilogy(np.array(fista_vals) - f_star + 1e-15, lw=2, label='FISTA $O(1/T^2)$', color='#55A868')
T = np.arange(1, 301)
ax.semilogy(T, 5.0 / T, ':', color='#C44E52', alpha=0.5, label=r'$\propto 1/T$')
ax.semilogy(T, 5.0 / T**2, ':', color='#55A868', alpha=0.5, label=r'$\propto 1/T^2$')
ax.set_xlabel('iteration'); ax.set_ylabel(r'$F(\mathbf{w}_t) - F^\star$')
ax.set_title('ISTA vs FISTA convergence on Lasso')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 🐍 Implementation from Scratch

We implement the full non-smooth optimization stack from first principles:

1. **Subgradient descent** for $\min_{\mathbf{w}} \|A\mathbf{w} - \mathbf{b}\|_1$ (L1
   regression) — demonstrating the non-monotone objective and $O(1/\sqrt{T})$ convergence.
2. **Soft-thresholding** as a standalone function, verified against the proximal definition.
3. **ISTA for Lasso:** $\min_{\mathbf{w}} \frac{1}{2}\|X\mathbf{w} - \mathbf{y}\|^2
   + \lambda\|\mathbf{w}\|_1$ — gradient step on the smooth loss, then soft-threshold.
4. **FISTA** — adding Nesterov momentum to ISTA. We compare convergence rates.
5. **Sparsity verification:** show that the ISTA/FISTA solution has exact zeros while the
   Ridge solution (from `04.03`'s smooth theory) does not.

In [ ]:
# ── 1. Subgradient descent for L1 regression: min ||Aw - b||_1 ──
rng = np.random.default_rng(0)
n, d = 80, 10
A = rng.normal(size=(n, d))
w_true_sub = rng.standard_normal(d)
b = A @ w_true_sub + rng.standard_normal(n) * 0.5  # noisy observations

def l1_regression_obj(w, A, b):
    return np.sum(np.abs(A @ w - b))

def l1_regression_subgrad(w, A, b):
    residual = A @ w - b
    return A.T @ np.sign(residual)  # subgradient of ||r||_1 is A^T sign(r)

w_sub = np.zeros(d)
sub_vals = []
best_val, best_w = np.inf, w_sub.copy()
for t in range(1, 501):
    val = l1_regression_obj(w_sub, A, b)
    sub_vals.append(val)
    if val < best_val:
        best_val, best_w = val, w_sub.copy()
    eta_t = 2.0 / np.sqrt(t)  # diminishing step size
    g_t = l1_regression_subgrad(w_sub, A, b)
    w_sub = w_sub - eta_t * g_t / (np.linalg.norm(g_t) + 1e-12)  # normalized subgradient

# Verify against scipy (which uses smooth methods internally with L1 reformulated as LP)
from scipy.optimize import linprog
# L1 regression as LP: min 1^T u s.t. Aw - b <= u, -(Aw - b) <= u, u >= 0
c_lp = np.concatenate([np.zeros(d), np.ones(n)])
A_ub = np.block([[A, -np.eye(n)], [-A, -np.eye(n)]])
b_ub = np.concatenate([b, -b])
bounds = [(None, None)] * d + [(0, None)] * n
res = linprog(c_lp, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
opt_val = res.fun

print('=== Subgradient Descent for L1 Regression ===')
print(f'Best subgradient obj: {best_val:.4f}')
print(f'Optimal obj (LP):     {opt_val:.4f}')
print(f'Relative gap:         {(best_val - opt_val) / opt_val:.4%}')
print(f'Note: objective is NON-MONOTONE (can increase at individual steps)')
increases = sum(1 for i in range(1, len(sub_vals)) if sub_vals[i] > sub_vals[i-1])
print(f'Steps where f increased: {increases}/{len(sub_vals)-1}\n')

# ── 2. Soft-thresholding: verify against brute-force proximal definition ──
def soft_thresh(v, t):
    """Soft-thresholding: prox of t * ||.||_1"""
    return np.sign(v) * np.maximum(np.abs(v) - t, 0)

# Verify: solve the proximal problem numerically for a few test points
print('=== Soft-Thresholding Verification ===')
for v_test, lam_test in [(2.5, 1.0), (-0.3, 0.5), (0.8, 0.8), (-1.5, 0.2)]:
    analytical = soft_thresh(v_test, lam_test)
    # Numerical: minimize |x| + (x-v)^2 / (2*lam) over a grid
    xs = np.linspace(-5, 5, 10000)
    obj = np.abs(xs) + (xs - v_test)**2 / (2 * lam_test)
    numerical = xs[np.argmin(obj)]
    print(f'  v={v_test:+.1f}, lambda={lam_test}: analytical={analytical:+.4f}, '
          f'numerical={numerical:+.4f}, match={abs(analytical - numerical) < 0.01}')

# ── 3. ISTA for Lasso ──
rng = np.random.default_rng(1)
n, d = 150, 80
X = rng.normal(size=(n, d))
w_true = np.zeros(d)
w_true[:5] = [3.0, -2.5, 1.8, -1.2, 0.7]  # 5 nonzero out of 80
y = X @ w_true + rng.normal(scale=0.3, size=n)
lam = 0.3

L = np.linalg.eigvalsh(X.T @ X / n).max()
eta = 1.0 / L

def lasso_obj(w):
    return 0.5 * np.mean((X @ w - y)**2) + lam * np.linalg.norm(w, 1)

def smooth_grad(w):
    return X.T @ (X @ w - y) / n

# ISTA
w_ista = np.zeros(d)
ista_history = []
for _ in range(500):
    ista_history.append(lasso_obj(w_ista))
    w_ista = soft_thresh(w_ista - eta * smooth_grad(w_ista), eta * lam)

# ── 4. FISTA ──
w_fista = np.zeros(d)
w_fista_prev = w_fista.copy()
t_k = 1.0
fista_history = []
for k in range(500):
    fista_history.append(lasso_obj(w_fista))
    t_next = (1 + np.sqrt(1 + 4 * t_k**2)) / 2
    momentum = (t_k - 1) / t_next
    y_k = w_fista + momentum * (w_fista - w_fista_prev)
    w_fista_prev = w_fista.copy()
    w_fista = soft_thresh(y_k - eta * smooth_grad(y_k), eta * lam)
    t_k = t_next

print(f'\n=== ISTA vs FISTA on Lasso (n={n}, d={d}, sparsity=5/{d}) ===')
print(f'ISTA  final obj: {ista_history[-1]:.6f}  (after 500 steps)')
print(f'FISTA final obj: {fista_history[-1]:.6f}  (after 500 steps)')

# ── 5. Sparsity: ISTA/FISTA vs Ridge ──
# Ridge (closed form): w_ridge = (X^T X + n*lam*I)^{-1} X^T y
w_ridge = np.linalg.solve(X.T @ X + n * lam * np.eye(d), X.T @ y)

print(f'\n=== Sparsity Comparison ===')
print(f'{"Method":12s} {"Exact zeros":>12s} {"Near-zero (<0.01)":>18s}')
for name, w in [('ISTA', w_ista), ('FISTA', w_fista), ('Ridge', w_ridge)]:
    exact = np.sum(w == 0)
    near = np.sum(np.abs(w) < 0.01)
    print(f'{name:12s} {exact:12d} {near:18d}')
print(f'\nGround truth has {np.sum(w_true != 0)} nonzero entries out of {d}.')
print('ISTA/FISTA produce EXACT zeros (L1 proximal sets small coords to 0);')
print('Ridge merely shrinks all coordinates — no exact sparsity.')

# Show recovered coefficients side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for ax, (name, w) in zip(axes, [('FISTA (L1)', w_fista), ('Ridge (L2)', w_ridge),
                                  ('Ground truth', w_true)]):
    colors = ['#C44E52' if abs(wi) > 0.01 else '#CCCCCC' for wi in w]
    ax.bar(range(d), w, color=colors, width=0.8)
    ax.set_title(f'{name}: {np.sum(np.abs(w) > 0.01)} nonzero')
    ax.set_xlabel('coefficient index')
axes[0].set_ylabel('weight value')
plt.tight_layout(); plt.show()

## 🤖 Why This Matters for AI

The proximal operator framework is not an abstract curiosity — it is the mathematical engine
behind a wide range of practical ML systems:

- **`sklearn.linear_model.Lasso`** uses coordinate descent (`04.11`) for its default solver,
  but the theoretical backbone is proximal gradient descent. The coordinate-wise update is
  *exactly* the soft-thresholding operator applied one coordinate at a time.

- **L1 regularization in neural networks** (sparse training, lottery ticket hypothesis)
  applies proximal operators *inside* the optimizer loop: after each SGD/Adam step, a
  soft-thresholding pass zeroes out small weights, gradually pruning the network.

- **Group Lasso** for structured sparsity (prune entire neurons, attention heads, or
  convolutional filters) generalizes soft-thresholding to block-wise proximal operators:
  $\operatorname{prox}_{\lambda\|\cdot\|_2}(\mathbf{v}) =
  \mathbf{v}\max(1 - \lambda/\|\mathbf{v}\|_2,\, 0)$ — the entire block is set to zero or
  shrunk uniformly.

- **The composite structure** $f + g$ (smooth data-fit + non-smooth regularizer) appears
  everywhere: least squares + L1 (Lasso), logistic loss + elastic net, reconstruction +
  total variation (image denoising), matrix factorization + nuclear norm.

Below we verify our FISTA implementation against `sklearn`'s production Lasso solver, then
demonstrate sparse recovery: recovering a few nonzero signals from noisy, high-dimensional
measurements — the core problem in compressed sensing, feature selection, and interpretable ML.

In [ ]:
from sklearn.linear_model import Lasso, Ridge

# ── 1. Cross-validate against sklearn's Lasso ──
# sklearn uses alpha = lambda (our lam), and its objective is
# (1/(2n))||Xw-y||^2 + alpha*||w||_1  — same as ours.
sk_lasso = Lasso(alpha=lam, fit_intercept=False, max_iter=5000, tol=1e-8)
sk_lasso.fit(X, y)

print('=== FISTA vs sklearn Lasso ===')
print(f'FISTA  obj: {lasso_obj(w_fista):.8f}  nonzeros: {np.sum(np.abs(w_fista) > 1e-6)}')
fista_sk_obj = 0.5 * np.mean((X @ sk_lasso.coef_ - y)**2) + lam * np.linalg.norm(sk_lasso.coef_, 1)
print(f'sklearn obj: {fista_sk_obj:.8f}  nonzeros: {np.sum(np.abs(sk_lasso.coef_) > 1e-6)}')
print(f'Max coef difference: {np.max(np.abs(w_fista - sk_lasso.coef_)):.6f}')
print(f'Solutions match: {np.allclose(w_fista, sk_lasso.coef_, atol=0.05)}')

# ── 2. Sparse recovery demo ──
# Generate a sparse ground truth with k nonzeros out of d dimensions
# and recover it from n < d measurements (compressed sensing regime)
rng = np.random.default_rng(42)
d_cs, k_cs, n_cs = 200, 8, 80  # d >> n: underdetermined
X_cs = rng.normal(size=(n_cs, d_cs)) / np.sqrt(n_cs)  # measurement matrix
w_cs_true = np.zeros(d_cs)
support = rng.choice(d_cs, k_cs, replace=False)
w_cs_true[support] = rng.standard_normal(k_cs) * 2
y_cs = X_cs @ w_cs_true + rng.normal(scale=0.05, size=n_cs)  # noisy measurements

# Solve with FISTA at several lambda values and track support recovery
L_cs = np.linalg.eigvalsh(X_cs.T @ X_cs / n_cs).max()
eta_cs = 1.0 / L_cs
lambdas = np.logspace(-3, 0, 30)
n_nonzeros = []
recovery_f1 = []

for lam_cs in lambdas:
    w = np.zeros(d_cs)
    w_prev = w.copy()
    tk = 1.0
    for _ in range(1000):
        tn = (1 + np.sqrt(1 + 4*tk**2)) / 2
        yk = w + (tk - 1)/tn * (w - w_prev)
        grad = X_cs.T @ (X_cs @ yk - y_cs) / n_cs
        w_prev = w.copy()
        w = soft_thresh(yk - eta_cs * grad, eta_cs * lam_cs)
        tk = tn
    nnz = np.sum(np.abs(w) > 1e-4)
    n_nonzeros.append(nnz)
    # F1 score for support recovery
    recovered = set(np.where(np.abs(w) > 1e-4)[0])
    true_support = set(support)
    tp = len(recovered & true_support)
    precision = tp / max(len(recovered), 1)
    recall = tp / len(true_support)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    recovery_f1.append(f1)

best_idx = np.argmax(recovery_f1)
print(f'\n=== Sparse Recovery (d={d_cs}, k={k_cs}, n={n_cs}) ===')
print(f'Best F1 = {recovery_f1[best_idx]:.3f} at lambda = {lambdas[best_idx]:.4f} '
      f'({n_nonzeros[best_idx]} nonzeros recovered, {k_cs} true)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.semilogx(lambdas, n_nonzeros, 'o-', color='#4C72B0', lw=1.5, markersize=4)
ax1.axhline(k_cs, color='#C44E52', ls='--', label=f'true sparsity k={k_cs}')
ax1.set_xlabel(r'$\lambda$'); ax1.set_ylabel('nonzero coefficients')
ax1.set_title('Regularization path: sparsity vs $\\lambda$')
ax1.legend(fontsize=9)

ax2.semilogx(lambdas, recovery_f1, 's-', color='#55A868', lw=1.5, markersize=4)
ax2.axvline(lambdas[best_idx], color='#C44E52', ls=':', alpha=0.7,
            label=f'best $\\lambda$={lambdas[best_idx]:.4f}')
ax2.set_xlabel(r'$\lambda$'); ax2.set_ylabel('F1 score (support recovery)')
ax2.set_title('Support recovery quality across $\\lambda$')
ax2.legend(fontsize=9)

plt.tight_layout(); plt.show()

print(f'\nAt the optimal lambda, FISTA recovers the true sparse support with F1={recovery_f1[best_idx]:.3f}.')
print('This is the core mechanism behind compressed sensing, feature selection, and')
print('interpretable ML: L1 proximal methods find sparse solutions in high dimensions.')

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Compute the subdifferential $\partial f(0)$ for each of the following:
   (a) $f(x) = |x|$, (b) $f(x) = \max(0, x)$ (ReLU), (c) $f(x) = \max(|x| - 1,\, 0)$
   (the "dead-zone" penalty). For each, verify the optimality condition: is
   $0 \in \partial f(0)$?

<details>
<summary>💡 Solution</summary>

(a) $f(x) = |x|$: at $x=0$, any $g \in [-1,1]$ satisfies $|y| \ge 0 + g \cdot y$ for all
$y$, so $\partial f(0) = [-1, 1]$. Since $0 \in [-1,1]$, $x=0$ is a global minimizer. ✓

(b) $f(x) = \max(0,x)$: for $x>0$, $f'(x)=1$; for $x<0$, $f'(x)=0$. At $x=0$: any
$g \in [0,1]$ satisfies $\max(0,y) \ge 0 + g \cdot y$ for all $y$ (check $y>0$: $y \ge gy$
requires $g \le 1$; check $y<0$: $0 \ge gy$ requires $g \ge 0$). So $\partial f(0) = [0, 1]$.
Since $0 \in [0,1]$, $x=0$ is a minimizer (as is every $x \le 0$). ✓

(c) $f(x) = \max(|x|-1, 0)$: at $x=0$, $f(0) = \max(-1,0) = 0$. For any $|y| \le 1$,
$f(y) = 0$, so $f(y) \ge 0 + g \cdot y$ requires $g \cdot y \le 0$ for all $|y| \le 1$,
which forces $g = 0$. For $|y| > 1$: $|y|-1 \ge g \cdot y$ with $g=0$ gives $|y|-1 \ge 0$,
true. So $\partial f(0) = \{0\}$, and $0 \in \{0\}$ confirms $x=0$ is a minimizer (along with
the entire interval $[-1,1]$). ✓

</details>

### 💭 UNDERSTAND
2. Subgradient descent can *increase* $f$ at individual steps — the Implementation cell
   shows this explicitly. Why doesn't this prevent convergence? Why are diminishing step sizes
   $\eta_t \to 0$ required, and why must $\sum \eta_t = \infty$?

<details>
<summary>💡 Solution</summary>

Progress in subgradient descent is measured in **distance to the optimum**
$\|\mathbf{w}_t - \mathbf{w}^\star\|$, not function value. The key inequality is:
$\|\mathbf{w}_{t+1} - \mathbf{w}^\star\|^2 \le \|\mathbf{w}_t - \mathbf{w}^\star\|^2
- 2\eta_t(f(\mathbf{w}_t) - f^\star) + \eta_t^2\|\mathbf{g}_t\|^2$.
The middle term pulls us closer (since $f(\mathbf{w}_t) \ge f^\star$), but the last term
$\eta_t^2 G^2$ pushes us away — it's the "noise" from using a subgradient instead of a
gradient. For the convergence sum to work: $\eta_t \to 0$ ensures the noise term vanishes,
while $\sum \eta_t = \infty$ ensures we take enough total step length to actually reach
$\mathbf{w}^\star$ from any starting point. A fixed $\eta$ makes the noise term constant,
causing perpetual oscillation with radius $\sim \eta G$ around the optimum — never converging.
The standard choice $\eta_t = c/\sqrt{t}$ satisfies both conditions and yields the
$O(1/\sqrt{T})$ rate.

</details>

### ✏️ DERIVE
3. Derive the soft-thresholding operator from the proximal definition for
   $g(x) = \lambda|x|$ (scalar case). Set the subdifferential of the proximal objective
   $h(x) = \lambda|x| + \frac{1}{2}(x - v)^2$ to contain zero (the optimality condition
   $0 \in \partial h(x^\star)$) and solve the three cases $x^\star > 0$, $x^\star < 0$,
   $x^\star = 0$ separately. Then verify your formula matches the Implementation cell's
   `soft_thresh` function.

<details>
<summary>💡 Solution outline</summary>

The proximal objective is $h(x) = \lambda|x| + \frac{1}{2}(x-v)^2$. Its subdifferential is
$\partial h(x) = \lambda\,\partial|x| + (x - v)$.

**Case 1: $x^\star > 0$.** $\partial|x^\star| = \{1\}$, so $0 = \lambda + x^\star - v$,
giving $x^\star = v - \lambda$. Valid when $v - \lambda > 0$, i.e. $v > \lambda$.

**Case 2: $x^\star < 0$.** $\partial|x^\star| = \{-1\}$, so $0 = -\lambda + x^\star - v$,
giving $x^\star = v + \lambda$. Valid when $v + \lambda < 0$, i.e. $v < -\lambda$.

**Case 3: $x^\star = 0$.** $\partial|0| = [-1,1]$, so $0 \in \lambda[-1,1] + (0-v)$
requires $|v| \le \lambda$.

Combining: $\operatorname{prox}_{\lambda|\cdot|}(v) = \operatorname{sign}(v)
\max(|v| - \lambda,\, 0) = \mathcal{S}_\lambda(v)$. This matches `soft_thresh(v, lam)`
exactly. The extension to $\mathbb{R}^d$ is immediate since
$\|\mathbf{w}\|_1 = \sum_j |w_j|$ is separable.

</details>

### 🐍 IMPLEMENT
4. Implement FISTA for elastic net:
   $\min_{\mathbf{w}} \frac{1}{2n}\|X\mathbf{w} - \mathbf{y}\|^2
   + \lambda_1\|\mathbf{w}\|_1 + \frac{\lambda_2}{2}\|\mathbf{w}\|^2_2$.
   Hint: the L2 term is smooth — absorb it into $f$, so
   $f(\mathbf{w}) = \frac{1}{2n}\|X\mathbf{w}-\mathbf{y}\|^2
   + \frac{\lambda_2}{2}\|\mathbf{w}\|^2$ with gradient
   $\nabla f = \frac{1}{n}X^\top(X\mathbf{w}-\mathbf{y}) + \lambda_2\mathbf{w}$ and
   smoothness constant $L = \|X^\top X\|/n + \lambda_2$. The proximal step is still
   soft-thresholding with $g = \lambda_1\|\cdot\|_1$.

<details>
<summary>✅ How to know you're right</summary>

Compare against `sklearn.linear_model.ElasticNet(alpha=lambda_1+lambda_2,
l1_ratio=lambda_1/(lambda_1+lambda_2))`. Your coefficients should match to within $10^{-3}$
after 1000 FISTA steps for moderate-sized problems ($n=200, d=100$).

</details>

### 🤖 APPLY
5. Generate a sparse ground-truth $\mathbf{w}^\star$ with 5 nonzero entries out of 100
   dimensions. Solve the Lasso with FISTA at 50 logarithmically-spaced $\lambda$ values from
   $\lambda_{\max}$ (the smallest $\lambda$ giving $\mathbf{w} = \mathbf{0}$; show that
   $\lambda_{\max} = \|X^\top\mathbf{y}\|_\infty / n$) down to $\lambda_{\max}/1000$. Plot
   the **regularization path** (each coefficient $w_j$ vs $\lambda$) and identify the
   $\lambda$ that recovers exactly 5 nonzero coordinates.

<details>
<summary>✅ What you should observe</summary>

At $\lambda = \lambda_{\max}$, all coefficients are exactly zero (the subgradient optimality
condition $\|X^\top(\mathbf{y} - X\mathbf{w})/n\|_\infty \le \lambda$ is satisfied at
$\mathbf{w}=0$ precisely when $\lambda \ge \|X^\top\mathbf{y}\|_\infty/n$). As $\lambda$
decreases, coefficients "enter" the model one by one — the first to appear is the one whose
column of $X$ has the largest correlation with $\mathbf{y}$, matching the intuition from
`13.03`'s regularization theory. The 5-nonzero ground truth should be recoverable at some
intermediate $\lambda$.

</details>

### 🚀 CHALLENGE
6. Prove that the proximal operator is a **firmly non-expansive** mapping:
   $$\|\operatorname{prox}_{\lambda g}(\mathbf{x})
   - \operatorname{prox}_{\lambda g}(\mathbf{y})\|^2
   + \|(\mathbf{x} - \operatorname{prox}_{\lambda g}(\mathbf{x}))
   - (\mathbf{y} - \operatorname{prox}_{\lambda g}(\mathbf{y}))\|^2
   \;\le\; \|\mathbf{x} - \mathbf{y}\|^2$$
   *Hint:* Let $\mathbf{p} = \operatorname{prox}(\mathbf{x})$,
   $\mathbf{q} = \operatorname{prox}(\mathbf{y})$. The optimality condition for $\mathbf{p}$
   is $\frac{1}{\lambda}(\mathbf{x} - \mathbf{p}) \in \partial g(\mathbf{p})$, and similarly
   for $\mathbf{q}$. Use the monotonicity of the subdifferential:
   $(\mathbf{u} - \mathbf{v})^\top(\mathbf{p} - \mathbf{q}) \ge 0$ for
   $\mathbf{u} \in \partial g(\mathbf{p})$, $\mathbf{v} \in \partial g(\mathbf{q})$.
   Then use firm non-expansiveness to prove ISTA's $O(1/T)$ convergence rate.

<details>
<summary>🔍 Solution outline</summary>

Let $\mathbf{p} = \operatorname{prox}(\mathbf{x})$,
$\mathbf{q} = \operatorname{prox}(\mathbf{y})$. From optimality:
$\mathbf{u} = (\mathbf{x}-\mathbf{p})/\lambda \in \partial g(\mathbf{p})$ and
$\mathbf{v} = (\mathbf{y}-\mathbf{q})/\lambda \in \partial g(\mathbf{q})$.
Monotonicity of $\partial g$: $(\mathbf{u}-\mathbf{v})^\top(\mathbf{p}-\mathbf{q}) \ge 0$.
Substituting: $\frac{1}{\lambda}[(\mathbf{x}-\mathbf{p})-(\mathbf{y}-\mathbf{q})]^\top
(\mathbf{p}-\mathbf{q}) \ge 0$. Expand
$\|\mathbf{x}-\mathbf{y}\|^2 = \|(\mathbf{p}-\mathbf{q})
+ [(\mathbf{x}-\mathbf{p})-(\mathbf{y}-\mathbf{q})]\|^2
= \|\mathbf{p}-\mathbf{q}\|^2
+ 2(\mathbf{p}-\mathbf{q})^\top[(\mathbf{x}-\mathbf{p})-(\mathbf{y}-\mathbf{q})]
+ \|(\mathbf{x}-\mathbf{p})-(\mathbf{y}-\mathbf{q})\|^2$.
The cross-term is $\ge 0$ by monotonicity, so
$\|\mathbf{x}-\mathbf{y}\|^2 \ge \|\mathbf{p}-\mathbf{q}\|^2
+ \|(\mathbf{x}-\mathbf{p})-(\mathbf{y}-\mathbf{q})\|^2$. ∎

For the $O(1/T)$ convergence of ISTA: firm non-expansiveness implies
$\|\mathbf{w}_{t+1} - \mathbf{w}^\star\|^2 \le \|\mathbf{w}_t - \eta\nabla f(\mathbf{w}_t)
- (\mathbf{w}^\star - \eta\nabla f(\mathbf{w}^\star))\|^2$. Combine with the descent lemma
from `04.03` ($L$-smoothness of $f$) and telescope over $T$ steps to obtain
$F(\mathbf{w}_T) - F^\star \le L\|\mathbf{w}_0 - \mathbf{w}^\star\|^2 / (2T)$.

</details>

---

## 📚 Further Reading
- Parikh & Boyd, ["Proximal Algorithms"](https://web.stanford.edu/~boyd/papers/prox_algs.html) (2014) — the definitive survey
- Beck & Teboulle, ["A Fast Iterative Shrinkage-Thresholding Algorithm for Linear Inverse Problems"](https://doi.org/10.1137/080716542) (2009) — the FISTA paper
- Boyd & Vandenberghe, [*Convex Optimization*](https://web.stanford.edu/~boyd/cvxbook/), Ch. 6 (proximal operators)
- Bubeck, ["Convex Optimization: Algorithms and Complexity"](https://arxiv.org/abs/1405.4980) (2015)

---

*Next notebook: [`04.11` — Coordinate Descent and Convergence Rate Taxonomy](11_coordinate_descent_and_convergence_rate_taxonomy.ipynb)*